# Donloading, Uncompressing, aand Pa\rsing

Download .dem files

In [ ]:
import requests
import os
from pathlib import Path

# This is a match from early august, feel free to try something newer (but try not to go past 2 months)
match_id = 8930664368

# Create subfolder
download_folder = Path("/content/downloaded_files")
download_folder.mkdir(parents=True, exist_ok=True)

# OpenDota
url = f"https://api.opendota.com/api/matches/{match_id}"
response = requests.get(url)

if response.status_code != 200:
    print("Error:", response.text)
else:
    data = response.json()
    replay_url = data.get("replay_url")

    if replay_url:
        print("Downloading replay... This can take a while (file is usually 20-100MB)")
        replay_response = requests.get(replay_url, stream=True)

        filename = download_folder / f"match_{match_id}.dem.bz2"

        with open(filename, "wb") as f:
            for chunk in replay_response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"✅ Download complete! File saved as: {filename}")
        print(f"File size: {os.path.getsize(filename) / (1024*1024):.1f} MB")
    else:
        print("Replay URL not found. Match might be too old or not parsed.")

Uncompress

In [ ]:
import bz2
import shutil
from pathlib import Path

# The file is already downloaded with the correct .bz2 extension
bz2_file = f"/content/downloaded_files/match_{match_id}.dem.bz2"

# Create decompressed folder
decompressed_folder = Path("/content/decompressed_files")
decompressed_folder.mkdir(parents=True, exist_ok=True)

# Save the decompressed file in the decompressed subfolder
dem_file = decompressed_folder / Path(bz2_file.replace(".bz2", "")).name

print(f"Decompressing {bz2_file} → {dem_file}")

with bz2.open(bz2_file, "rb") as f_in:
    with open(dem_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("✅ Decompression complete!")
print(f"New file size: {Path(dem_file).stat().st_size / (1024*1024):.1f} MB")

Parsing

In [ ]:
!git clone https://github.com/skadistats/clarity-examples.git /content/clarity-examples

# We will also make sure the gradle wrapper is executable
!chmod +x /content/clarity-examples/gradlew

In [ ]:
!apt-get update
!apt-get install -y openjdk-17-jdk-headless -qq
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("✅ Java 17 installed and JAVA_HOME configured!")

In [ ]:
import subprocess
import os

dem_file = f"/content/decompressed_files/match_{match_id}.dem"
clarity_dir = "/content/clarity-examples"

# We discovered that tasks are formatted as {exampleName}Run
task_name = "infoRun"

print(f"Running Clarity example: {task_name} on {dem_file}\n")

if os.path.exists(clarity_dir):
    cmd = f'./gradlew {task_name} --args="{dem_file}"'
    print(f"Executing: {cmd}\n")

    process = subprocess.Popen(cmd, shell=True, cwd=clarity_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Print output line by line
    for i, line in enumerate(process.stdout):
        print(line.strip())
        # Truncate output if it gets too long
        if i > 100:
            print("...[Output Truncated]...")
            process.kill()
            break
else:
    print(f"Clarity directory not found at {clarity_dir}.")


In [ ]:
import subprocess
import os

dem_file = f"/content/decompressed_files/match_{match_id}.dem"
clarity_dir = "/content/clarity-examples"
task_name = "combatlogRun"
output_txt = "/content/combatlog.txt"

print(f"Extracting combat log using {task_name}... This might take a minute.")

if os.path.exists(clarity_dir):
    # We stream the output directly into a text file because it can be quite large
    cmd = f'./gradlew {task_name} --args="{dem_file}" > {output_txt}'

    # Run the process
    subprocess.run(cmd, shell=True, cwd=clarity_dir)

    if os.path.exists(output_txt):
        print(f"✅ Combat log saved to {output_txt}")
        print(f"File size: {os.path.getsize(output_txt) / (1024*1024):.2f} MB\n")

        print("--- First 15 lines of the combat log ---")
        with open(output_txt, "r") as f:
            for _ in range(15):
                print(f.readline().strip())
        print("...\n")
        print("Note: You can parse this text file into a pandas DataFrame/CSV using regular expressions to extract the time, attacker, target, and damage values.")
    else:
        print("Failed to generate combat log.")
else:
    print(f"Clarity directory not found at {clarity_dir}.")


In [ ]:
import pandas as pd
import re
import os

log_file = "/content/combatlog.txt"
parsed_data = []

# Regex pattern to match lines starting with a timestamp like [00:14:23.600]
# Group 1 captures the time, Group 2 captures the event message
log_pattern = re.compile(r"^\[(.*?)\]\s+(.*)$")

if os.path.exists(log_file):
    print("Parsing combat log into DataFrame...")
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            match = log_pattern.match(line.strip())
            if match:
                parsed_data.append({
                    "Time": match.group(1),
                    "Event": match.group(2)
                })

    # Convert the list of dictionaries to a pandas DataFrame
    df_combat = pd.DataFrame(parsed_data)

    print(f"Successfully parsed {len(df_combat)} events!")
    display(df_combat.head(10))
else:
    print(f"File not found: {log_file}")

In [ ]:
import os

examples_path = "/content/clarity-examples/src/main/java/skadistats/clarity/examples"
if os.path.exists(examples_path):
    examples = [d for d in os.listdir(examples_path) if os.path.isdir(os.path.join(examples_path, d))]
    print("Available Clarity Examples:")
    print(sorted(examples))

    # Check for keywords related to movement
    matches = [e for e in examples if 'pos' in e.lower() or 'move' in e.lower() or 'loc' in e.lower()]
    print(f"\nPotential matches for movement tracking: {matches}")
else:
    print("Examples directory not found.")

In [ ]:
import subprocess
import os

dem_file = "/content/decompressed_files/match_8866148821.dem"
clarity_dir = "/content/clarity-examples"
task_name = "positionRun"
output_txt = "/content/position.txt"

print(f"Extracting position data using {task_name}... This might take a minute.")

if os.path.exists(clarity_dir):
    # Stream the output directly into a text file
    cmd = f'./gradlew {task_name} --args="{dem_file}" > {output_txt}'

    # Run the process
    subprocess.run(cmd, shell=True, cwd=clarity_dir)

    if os.path.exists(output_txt):
        print(f"✅ Position data saved to {output_txt}")
        print(f"File size: {os.path.getsize(output_txt) / (1024*1024):.2f} MB\n")

        print("--- First 15 lines of the position log ---")
        with open(output_txt, "r") as f:
            for _ in range(15):
                print(f.readline().strip())
        print("...\n")
    else:
        print("Failed to generate position log.")
else:
    print(f"Clarity directory not found at {clarity_dir}.")

In [ ]:
import itertools

output_txt = "/content/position.txt"

print("--- Sample of position data logs ---")
with open(output_txt, "r") as f:
    # Skip the first 15 lines (which we already saw) and read the next 30
    for line in itertools.islice(f, 15, 45):
        print(line.strip())

In [ ]:
import pandas as pd
import re
import os

position_file = "/content/position.txt"
pos_data = []

# Regex pattern to match lines like: Player 04 changed position to [9301.844, 10240.531, 16768.0]
pos_pattern = re.compile(r"Player (\d+) changed position to \[([\d\.\-]+),\s*([\d\.\-]+),\s*([\d\.\-]+)\]")

if os.path.exists(position_file):
    print("Parsing position log into DataFrame...")
    with open(position_file, "r", encoding="utf-8") as f:
        for line in f:
            match = pos_pattern.search(line)
            if match:
                pos_data.append({
                    "Player": int(match.group(1)),
                    "X": float(match.group(2)),
                    "Y": float(match.group(3)),
                    "Z": float(match.group(4))
                })

    # Convert the list of dictionaries to a pandas DataFrame
    df_positions = pd.DataFrame(pos_data)

    print(f"Successfully parsed {len(df_positions)} position updates!")
    display(df_positions.head(10))
else:
    print(f"File not found: {position_file}")

In [ ]:
import os

position_dir = "/content/clarity-examples/src/main/java/skadistats/clarity/examples/position"
java_files = [f for f in os.listdir(position_dir) if f.endswith('.java')]

print(f"Java files found: {java_files}\n")

for java_file in java_files:
    file_path = os.path.join(position_dir, java_file)
    print(f"--- {java_file} ---")
    with open(file_path, "r") as f:
        print(f.read())
    print("\n")

In [ ]:
import os

file_path = "/content/clarity-examples/src/main/java/skadistats/clarity/examples/position/Main.java"

# Let's find the exact lines where the position is being printed
with open(file_path, "r") as f:
    lines = f.readlines()

print("Relevant lines from Main.java:\n")
for i, line in enumerate(lines):
    if "changed position" in line or "System.out" in line or "log.info" in line:
        # Print surrounding context (5 lines before and after)
        start = max(0, i - 5)
        end = min(len(lines), i + 6)
        for j in range(start, end):
            print(f"{j}: {lines[j].rstrip()}")
        print("-" * 40)

In [ ]:
import os

file_path = "/content/clarity-examples/src/main/java/skadistats/clarity/examples/position/Main.java"

with open(file_path, "r") as f:
    content = f.read()

# Import the Context class
if "import skadistats.clarity.processor.runner.Context;" not in content:
    content = content.replace(
        "import skadistats.clarity.processor.runner.SimpleRunner;",
        "import skadistats.clarity.processor.runner.SimpleRunner;\nimport skadistats.clarity.processor.runner.Context;"
    )

# Inject Context into the class
if "private Context ctx;" not in content:
    content = content.replace(
        "private Entities entities;",
        "private Entities entities;\n\n    @Insert\n    private Context ctx;"
    )

# Modify the print statement to include ctx.getTick()
original_print = 'System.out.format("Player %02d changed position to %s\\n", p, newPosition.toString());'
new_print = 'System.out.format("[%d] Player %02d changed position to %s\\n", ctx.getTick(), p, newPosition.toString());'
content = content.replace(original_print, new_print)

with open(file_path, "w") as f:
    f.write(content)

print("Modified Main.java to include the game tick! Re-running position extraction...")

import subprocess
output_txt_with_ticks = "/content/position_with_ticks.txt"
cmd = f'./gradlew positionRun --args="{dem_file}" > {output_txt_with_ticks}'
subprocess.run(cmd, shell=True, cwd=clarity_dir)

print(f"✅ New position data saved to {output_txt_with_ticks}")
with open(output_txt_with_ticks, "r") as f:
    for _ in range(25):
        print(f.readline().strip())

In [ ]:
import pandas as pd
import re
import os

position_with_ticks_file = "/content/position_with_ticks.txt"
pos_data_ticks = []

# Regex pattern to match lines like: [25670] Player 04 changed position to [9301.844, 10240.531, 16768.0]
pos_pattern_ticks = re.compile(r"\[(\d+)\] Player (\d+) changed position to \[([\d\.\-]+),\s*([\d\.\-]+),\s*([\d\.\-]+)\]")

if os.path.exists(position_with_ticks_file):
    print("Parsing position log with ticks into DataFrame...")
    with open(position_with_ticks_file, "r", encoding="utf-8") as f:
        for line in f:
            match = pos_pattern_ticks.search(line)
            if match:
                pos_data_ticks.append({
                    "Tick": int(match.group(1)),
                    "Player": int(match.group(2)),
                    "X": float(match.group(3)),
                    "Y": float(match.group(4)),
                    "Z": float(match.group(5))
                })

    # Convert the list of dictionaries to a pandas DataFrame
    df_positions_ticks = pd.DataFrame(pos_data_ticks)

    print(f"Successfully parsed {len(df_positions_ticks)} position updates with ticks!")
    display(df_positions_ticks.head(10))
else:
    print(f"File not found: {position_with_ticks_file}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter out extreme outliers if any exist to get a clean map view,
# but for standard Dota 2 maps, plotting X vs Y works well.
plt.figure(figsize=(12, 12))

# Create a scatter plot of X vs Y, color-coded by Player ID
sns.scatterplot(
    data=df_positions_ticks,
    x="X",
    y="Y",
    hue="Player",
    palette="tab10",
    s=10,
    alpha=0.1,
    edgecolor=None
)

plt.title("Dota 2 Player Movement Trajectories", fontsize=16)
plt.xlabel("X Coordinate", fontsize=12)
plt.ylabel("Y Coordinate", fontsize=12)

# Adjust legend
plt.legend(title="Player ID", bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Increase the embed limit for large animations (default is 20MB)
plt.rcParams['animation.embed_limit'] = 50.0

# 1. Clean the data: handle potential multiple updates per tick by taking the last one
# Then unstack so we have a MultiIndex of [X, Y] for each Player
df_clean = df_positions_ticks.groupby(['Tick', 'Player'])[['X', 'Y']].last().unstack('Player')

# 2. Define the sample interval
# Dota 2 runs at 30 ticks per second. 5 seconds = 150 ticks
sample_interval = 150
min_tick = df_positions_ticks['Tick'].min()
max_tick = df_positions_ticks['Tick'].max()
target_ticks = np.arange(min_tick, max_tick + sample_interval, sample_interval)

# 3. Create a continuous timeline and forward-fill positions
# This ensures players who stand still and don't emit updates remain on the map
union_index = df_clean.index.union(target_ticks).sort_values()
df_sampled = df_clean.reindex(union_index).ffill().bfill().loc[target_ticks]

# 4. Set up the figure for animation
fig, ax = plt.subplots(figsize=(10, 10))
x_min, x_max = df_positions_ticks['X'].min() - 500, df_positions_ticks['X'].max() + 500
y_min, y_max = df_positions_ticks['Y'].min() - 500, df_positions_ticks['Y'].max() + 500
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_title("Dota 2 Player Movement - Animated (5s intervals)", fontsize=16)
ax.set_xlabel("X Coordinate", fontsize=12)
ax.set_ylabel("Y Coordinate", fontsize=12)
ax.grid(True, alpha=0.3)

# Prepare colors for the 10 players
colors = plt.cm.tab10(np.arange(10))

# Initialize with the first frame's positions to avoid size mismatch with colors array
x_init = df_sampled.loc[target_ticks[0], 'X'].values
y_init = df_sampled.loc[target_ticks[0], 'Y'].values
scat = ax.scatter(x_init, y_init, s=120, c=colors, edgecolors='black', linewidths=0.5)

# Time annotation
time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, fontsize=14,
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Fixed legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=10) for i in range(10)]
ax.legend(handles, [f'Player {i}' for i in range(10)], title="Player ID", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# 5. Animation functions
def init():
    time_text.set_text('')
    return scat, time_text

def update(frame_idx):
    tick = target_ticks[frame_idx]

    # Extract X and Y arrays for this specific frame
    x_vals = df_sampled.loc[tick, 'X'].values
    y_vals = df_sampled.loc[tick, 'Y'].values

    offsets = np.column_stack((x_vals, y_vals))
    scat.set_offsets(offsets)

    # Calculate time based on tick (30 ticks/sec)
    seconds = int((tick - min_tick) / 30)
    mins = seconds // 60
    secs = seconds % 60
    time_text.set_text(f'Time: {mins:02d}:{secs:02d}')

    return scat, time_text

# 6. Generate and display the animation
print(f"Generating animation with {len(target_ticks)} frames... This may take a minute.")
anim = FuncAnimation(fig, update, frames=len(target_ticks), init_func=init, blit=True, interval=100)

# Close the static plot so it doesn't double-render
plt.close()

# Display the interactive HTML player
display(HTML(anim.to_jshtml()))

In [ ]:
import subprocess
import os

dem_file = "/content/decompressed_files/match_8866148821.dem"
clarity_dir = "/content/clarity-examples"
task_name = "serializersRun"
output_txt = "/content/available_fields.txt"

print(f"Extracting available fields and entity structures using {task_name}... This might take a moment.")

if os.path.exists(clarity_dir):
    # Stream the output directly into a text file
    cmd = f'./gradlew {task_name} --args="{dem_file}" > {output_txt}'

    # Run the process
    subprocess.run(cmd, shell=True, cwd=clarity_dir)

    if os.path.exists(output_txt):
        print(f"\n✅ Available fields saved to {output_txt}")
        print(f"File size: {os.path.getsize(output_txt) / (1024*1024):.2f} MB\n")

        print("--- Sample of available entities and their fields ---")
        with open(output_txt, "r", encoding="utf-8", errors="ignore") as f:
            for _ in range(50):
                print(f.readline().strip())
        print("...\n\nNote: Open the 'available_fields.txt' file from the Colab file browser to see all available fields for every entity type!")
    else:
        print("Failed to generate fields log.")
else:
    print(f"Clarity directory not found at {clarity_dir}.")

In [ ]:
import glob
import os

clarity_dir = "/content/clarity-examples"
# Find the flattables text file generated by the serializersRun task
flattables_files = glob.glob(os.path.join(clarity_dir, "flattables_*.txt"))

if flattables_files:
    flattables_file = flattables_files[0]
    print(f"Found fields file: {flattables_file}")
    print(f"File size: {os.path.getsize(flattables_file) / (1024*1024):.2f} MB\n")

    print("--- Sample of available entities and their fields ---")
    with open(flattables_file, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(100):
            print(f.readline().strip())
    print("...\n\nNote: You can open the full file from the Colab file browser to explore all fields.")
else:
    print("Could not find the generated flattables file.")

In [ ]:
import glob
import os

clarity_dir = "/content/clarity-examples"
# Find the flattables text file generated by the serializersRun task
flattables_files = glob.glob(os.path.join(clarity_dir, "flattables_*.txt"))

if flattables_files:
    flattables_file = flattables_files[0]
    print(f"Found fields file: {flattables_file}")
    print(f"File size: {os.path.getsize(flattables_file) / (1024*1024):.2f} MB\n")

    print("--- Sample of available entities and their fields ---")
    with open(flattables_file, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(100):
            print(f.readline().strip())
    print("...\n\nNote: You can open the full file from the Colab file browser to explore all fields.")
else:
    print("Could not find the generated flattables file.")